In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch 
import numpy as np
import random
torch.autograd.set_detect_anomaly(True)
torch.multiprocessing.set_sharing_strategy("file_descriptor")
seed = 140421
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
from detectron2.data.datasets.pano360 import CalibDataset
from detectron2.data.datasets.pano360 import CameraRegressorDataset

debug = False
train_calib = CalibDataset(
    train=True,
    json_name="datasets/pano360_crops_dataset_cvpr_myDistWider_train.json",
    debug=debug,
)
val_calib = CalibDataset(
    train=False,
    json_name="datasets/pano360_crops_dataset_cvpr_myDistWider_train.json",
    debug=debug,
)

train_pano = CameraRegressorDataset(
    is_train=True,
    debug=debug,
)
val_pano = CameraRegressorDataset(
    is_train=False,
    debug=debug,
)

In [ ]:
import cv2
from torch import nn
import matplotlib.pyplot as plt
from detectron2.data.pano360_utils import ( 
    bins2pitch, bins2roll, bins2vfov, bins2horizon, showHorizonLineFromHorizon, showHorizonLine,
    pitch2soft_idx, roll2soft_idx, vfov2soft_idx, horizon2soft_idx,
    soft_idx_to_angle,vfov_bins, pitch_bins, roll_bins, horizon_bins, get_softargmax
)
from detectron2.utils.visualizer import Visualizer

if debug:
    print("Viewing samples")
    max_vis = 15
    for i, d in enumerate(train_calib):
        print(d.keys())
        img = cv2.imread(d["file_name"])
        visualizer = Visualizer(img[:, :, ::-1], scale=0.5)
        out = visualizer.get_output()
        img = out.get_image()
        gt_pitch    = d["pitch"]
        gt_roll     = d["roll"]
        gt_vfov     = d["vfov"]
        print("pitch", "roll", "vfov")
        print(gt_pitch, gt_roll, gt_vfov)
        print(np.degrees(gt_pitch), np.degrees(gt_roll), np.degrees(gt_vfov))
        # plt.imshow(img)
        gt_pitch = bins2pitch(nn.functional.one_hot(torch.as_tensor(d["logits"]["gt_pitch"]), num_classes=256).float())
        gt_roll = bins2roll(nn.functional.one_hot(torch.as_tensor(d["logits"]["gt_roll"]), num_classes=256).float())
        gt_vfov = bins2vfov(nn.functional.one_hot(torch.as_tensor(d["logits"]["gt_vfov"]), num_classes=256).float())
        print("pitch", "roll", "vfov")
        print(gt_pitch, gt_roll, gt_vfov)
        print(np.degrees(gt_pitch), np.degrees(gt_roll), np.degrees(gt_vfov))
        anno_img_logits, _ = showHorizonLine(img, gt_vfov, gt_pitch, gt_roll)
        # anno_img_logits = showHorizonLineFromHorizon(anno_img_logits, gt_horizon, color=(255, 255, 255), width=3, debug=True, GT=True, ) # White: GT horizon without roll

        gt_pitch = pitch2soft_idx(d["pitch"])
        gt_roll = roll2soft_idx(d["roll"])
        gt_vfov = vfov2soft_idx(d["vfov"])
        gt_vfov = soft_idx_to_angle(gt_vfov, min=np.min(vfov_bins), max=np.max(vfov_bins))
        gt_pitch = soft_idx_to_angle(gt_pitch, min=np.min(pitch_bins), max=np.max(pitch_bins))
        gt_roll = soft_idx_to_angle(gt_roll, min=-0.6, max=0.6)
        # gt_roll = soft_idx_to_angle(gt_roll, min=np.min(roll_bins), max=np.max(roll_bins))
        print("pitch", "roll", "vfov")
        print(gt_pitch, gt_roll, gt_vfov)
        print(np.degrees(gt_pitch), np.degrees(gt_roll), np.degrees(gt_vfov))

        anno_img, _ = showHorizonLine(img, gt_vfov, gt_pitch, gt_roll)
        # anno_img = showHorizonLineFromHorizon(anno_img, gt_horizon, color=(255, 255, 255), width=3, debug=True, GT=True, ) # White: GT horizon without roll
        # Side-by-side comparison
        comparison_img = np.concatenate([anno_img_logits, anno_img], axis=1)
        plt.imshow(comparison_img)
        plt.axis("off")
        plt.show()
        if max_vis == i:
            break

In [ ]:
debug_roll_hist = True
if debug_roll_hist:
    train_pitch = []
    train_roll = []
    train_vfov = []
    for i, d in enumerate(train_calib):
        train_roll.append(d["roll"])
        train_pitch.append(d["pitch"])
        train_vfov.append(d["vfov"])
    for v in [train_pitch, train_roll, train_vfov]:
        print(np.mean(v))
        print(np.std(v))
        print(np.min(v))
        print(np.max(v))
        plt.hist(v, bins=256)
        plt.show()

In [ ]:
debug_roll_hist = True
if debug_roll_hist:
    train_pitch = []
    train_roll = []
    train_vfov = []
    for i, d in enumerate(train_pano):
        train_roll.append(d["roll"])
        train_pitch.append(d["pitch"])
        train_vfov.append(d["vfov"])
    for v in [train_pitch, train_roll, train_vfov]:
        print(np.mean(v))
        print(np.std(v))
        print(np.min(v))
        print(np.max(v))
        plt.hist(v, bins=256)
        plt.show()

In [ ]:
from detectron2.data import DatasetCatalog
DatasetCatalog.register("Pano360_scale_train", train_calib)
DatasetCatalog.register("Pano360_scale_val", val_calib)
DatasetCatalog.register("Pano360_train", train_pano)
DatasetCatalog.register("Pano360_val", val_pano)

In [ ]:
import os
from detectron2.engine import CalibTrainer
from detectron2 import model_zoo
from detectron2.config import get_cfg
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "garbage_collection_threshold:0.6,max_split_size_mb:128,expandable_segments:True"
cfg = get_cfg()
config_path = "COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml"
experiment_name = "calib-only-1gpu-biasl2"
# experiment_name = "calib-only-1gpu"
cfg.OUTPUT_DIR = os.path.join("output", experiment_name)
cfg.merge_from_file(model_zoo.get_config_file(config_path))
# cfg.DATALOADER
cfg.DATASETS.TRAIN = ("Pano360_train",)
cfg.DATASETS.TEST = ("Pano360_val",)
cfg.DATALOADER.NUM_WORKERS = 4
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(config_path)  # Let training initialize from model zoo
cfg.SOLVER.IMS_PER_BATCH = 16  # Can go up to 12 in 12gb
cfg.SOLVER.BASE_LR = 0.02  # pick a good LR
cfg.MODEL.META_ARCHITECTURE = "CameraRCNN"
cfg.MODEL.PROPOSAL_GENERATOR.NAME = "PrecomputedProposals"  # Disable RPN Network. -- Must add proposals to input data
cfg.MODEL.KEYPOINT_ON=False
cfg.VIS_PERIOD = 500
cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS=False
cfg.SOLVER.STEPS = (13000, 17000)  # Same ratio as base config path
cfg.SOLVER.MAX_ITER = 20000  # One whole epoch in Pano360 ~187206/BATCH_SIZE
cfg.SOLVER.AMP.ENABLED = True  # Enable AMP here -- improve 10s per iter approx.
cfg.FLOAT32_PRECISION = "medium"

trainer = CalibTrainer(cfg) 
# NOTE: change value of resume if we have a last_checkpoint
trainer.resume_or_load(resume=True)

In [ ]:
# trainer.train()

In [ ]:
trainer.test(cfg, trainer.model)